# Chapter 16: The same beginning

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "16"
__file__ = os.path.abspath("run.py")

## The scene

Friday 31 July, 08:45. Varma's first question each morning takes the best part of a minute to
begin answering. The second, straight after, starts almost at once.

"Why is it slow only when I arrive?"

Case file 02 chapter 20 found one cause: a timestamp in the system prompt, new every request,
so that nothing after it could be reused.

The cause this morning is the nightly sync. At 01:58 it rewrites the case memory, the second
thing LANTERN reads.

"So every morning it reads everything again from the top," Okafor says.

"From the first thing that changed," you say.

## The concept

Before a model writes a word it reads the whole prompt. That reading is most of the wait on a
long context. A model server keeps the work it did on the last prompt, and when the next prompt
**begins** the same way it reuses that work and reads only from the first token that differs.
This is the **prompt cache**, or prefix cache.

So where a change sits decides what it costs. A change in the last line costs the last line. A
change in the first line costs everything. The rule that follows is a **layout**: what changes
least goes first, what changes most goes last.

LANTERN's sections change at different rates:

| Changes | Sections |
|---|---|
| rarely | standing orders, tool schemas |
| weekly | standing notes, the summaries written each Friday |
| nightly | the case memory, which the sync rewrites at 01:58 |
| per turn | the session so far |
| per question | the question |

Today the case memory is second. A layout for the cache puts it after everything weekly, so the
sync's change costs only what follows it.

On a hosted model the same idea is explicit: the request marks **cache breakpoints** at the end
of the stable groups, and the provider bills cached tokens at a fraction of the input price.

```mermaid
flowchart LR
    subgraph today[Today's order]
        A1[standing orders] --> A2[case memory, rewritten nightly] --> A3[notes and summaries] --> A4[tool schemas] --> A5[session] --> A6[question]
    end
    subgraph stable[Stable first]
        B1[standing orders] --> B2[tool schemas] --> B3[notes and summaries] --> B4[case memory] --> B5[session] --> B6[question]
    end
    A2 -.->|read again from here: 4,576 tokens| A6
    B4 -.->|read again from here: 1,150 tokens| B6
```

## How it works

1. **The same request, and a change at the start, the middle and the end.** A 5,410-token
   context sent five ways, three times each: the first time, the same again, with the question
   changed, with a line changed 2,562 tokens in, and with the first line changed. Each request
   asks for one token, so the time Ollama reports for the prompt (`Reply.prompt_seconds`) is
   the reading.
2. **The morning after the sync.** Yesterday's context is sent, then the case memory is
   rewritten as the sync does and the morning's first and second questions are asked, in
   today's order and stable first (`lantern.layout.layout`).
3. **Does the order change the answers?** The 26 signed questions asked again with the stable
   first layout, against chapter 02's baseline.
4. **On a hosted model: breakpoints and a price calculation.** The Anthropic request for the
   stable first layout is built with a breakpoint at the end of each group, and a ten-question
   session is priced from the measured token counts. No hosted call is made.

The layout is a stable sort and a list of where the groups end:

```python
ordered = sorted(parts, key=lambda p: CHANGES.index(changes[p.name]))
ends = [i for i in range(len(ordered) - 1)
        if changes[ordered[i].name] != changes[ordered[i + 1].name]]
return ordered, ends[:MAX_BREAKPOINTS]
```

On Ollama the chat template puts tool schemas straight after the system message, so in the
stable first order the system message holds only the standing orders, and every later section
travels as a user turn of its own (`lantern.layout.request`).

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json
import os
import statistics
import subprocess
import time
from dataclasses import replace

from lantern import baseline, context, layout, learn
from lantern.config import ROOT, load_config
from lantern.context import SYSTEM, Section
from lantern.llm import chat
from lantern.ui import Table, chapter, console, evidence, finding, note, save_shot, step

chapter("16", "The same beginning", """
Friday 31 July, 08:45. Varma's first question each morning takes the best part of a minute to
begin answering. The second, straight after, starts almost at once.

"Why is it slow only when I arrive?"

Case file 02 chapter 20 found one cause: a timestamp in the system prompt, new every request,
so that nothing after it could be reused.

The cause this morning is the nightly sync. At 01:58 it rewrites the case memory, the second
thing LANTERN reads.

"So every morning it reads everything again from the top," Okafor says.

"From the first thing that changed," you say.
""")

MINE = learn.mine()  # --mine: your layout() from practice/exercise.py
LAYOUT = MINE.layout if MINE else layout.layout
OUT = ROOT / ".scratch" / "16"
OUT.mkdir(parents=True, exist_ok=True)
REPEATS = 3  # every timing is taken this many times; the machine is shared
SESSION = 10  # questions in the priced session of step 5
cfg = load_config()
prices = cfg["hosted"]["prices"]


def others() -> int:
    """Other lessons running on this machine now, from any lab: they share the model server."""
    try:
        out = subprocess.run(["pgrep", "-f", r"lessons/[0-9][0-9]-.*/run\.py"],
                             capture_output=True, text=True, timeout=5).stdout.split()
    except (OSError, subprocess.SubprocessError):
        return -1
    return len([p for p in out if int(p) not in (os.getpid(), os.getppid())])


def send(parts, ordered=None):
    """One request that reads the prompt and writes one token: the time is the reading."""
    messages, schemas = layout.request(ordered if ordered is not None else parts)
    r = chat(messages, tools=schemas, seed=7, max_tokens=1)
    return {"seconds": round(r.prompt_seconds, 2), "tokens": r.prompt_tokens}


def stamped(parts, stamp):
    return [Section("session stamp", SYSTEM, f"Session opened {stamp}.\n"), *parts]


def changed_line(parts, name, old, new):
    out = []
    for s in parts:
        if s.name == name:
            if old not in s.text:
                raise ValueError(f"{old!r} is not in {name}")
            s = replace(s, text=s.text.replace(old, new, 1))
        out.append(s)
    return out


def before(parts, name):
    """Tokens of text before the start of a section, counted on the text."""
    i = [s.name for s in parts].index(name)
    return context.count_tokens(context.flat(parts[:i])) if i else 0


busy_start = others()
questions = baseline.questions()
base = context.sections()
names = [s.name for s in base]
sizes = {s.name: context.count_tokens(s.text) for s in base}
total = sum(sizes.values())
MIDDLE = "summary week-2026-06-29"
middle_at = before(base, MIDDLE)
other_q = questions[5]["question"]

### 1. The same request, and a change at the start, the middle and the end

In [ ]:
step("1. The same request, and a change at the start, the middle and the end")
evidence("what is sent, in order (tokens as qwen3 counts the text)",
         "\n".join(f"{n:<26}{sizes[n]:>6,}" for n in names) + f"\n{'all':<26}{total:>6,}")
p_count = learn.predict(
    "The same request is sent twice. What does Ollama report as the second request's prompt "
    "token count?", ["The same count as the first", "Only the tokens it had to read again",
                     "Zero"])
p_mid = learn.predict(
    f"A line {middle_at:,} tokens into a {total:,}-token context is changed. How long does "
    "reading the prompt take, against reading it the first time?",
    ["About the same", "Roughly the share after the change", "Almost nothing"])
rows: dict[str, list[dict]] = {k: [] for k in ("first time", "the same again", "end changed",
                                                "middle changed", "start changed")}
for rep in range(REPEATS):
    stamp = f"Friday 31 July 2026 08:45:{rep:02d}.{time.time_ns() % 10**6:06d}"
    first = stamped(base, stamp)
    rows["first time"].append(send(first))
    rows["the same again"].append(send(first))
    rows["end changed"].append(send(baseline.asking(first, other_q)))
    send(first)
    rows["middle changed"].append(send(changed_line(first, MIDDLE, "\n", " (rewritten)\n")))
    send(first)
    rows["start changed"].append(send(stamped(base, stamp + " again")))
where = {"first time": "nothing read before", "the same again": "none",
         "end changed": f"the question, after {total - sizes['the question']:,}",
         "middle changed": f"after {middle_at:,}", "start changed": "the first line"}
table = Table(title=f"Reading the prompt, qwen3:8b on local Ollama, {REPEATS} times each")
for col in ("request", "change", "prompt tokens", *[f"seconds, run {i + 1}"
                                                    for i in range(REPEATS)], "median"):
    table.add_column(col, justify="left" if col in ("request", "change") else "right")
med = {}
for k, rs in rows.items():
    med[k] = statistics.median(r["seconds"] for r in rs)
    table.add_row(k, where[k], f"{rs[0]['tokens']:,}", *[f"{r['seconds']:.2f}" for r in rs],
                  f"{med[k]:.2f}")
console.print(table)
same_count = all(r["tokens"] == rows["first time"][0]["tokens"] for rs in rows.values()
                 for r in rs)
note(f"The first time, {med['first time']:.2f} s. The same again, {med['the same again']:.2f} s. "
     f"Prompt tokens reported were the same on every request: {same_count}.")
learn.reveal(p_count, answer=0 if same_count else 1,
             why="Ollama reports the whole prompt in prompt_eval_count whatever it reused. The "
             "saving shows only in the time, prompt_eval_duration.")
ratio_mid = med["middle changed"] / max(med["first time"], 1e-9)
learn.reveal(p_mid, answer=0 if ratio_mid > 0.85 else (2 if ratio_mid < 0.15 else 1),
             why=f"The server reuses the prompt up to the first token that differs and reads "
             f"the rest again. {1 - middle_at / total:.0%} of this prompt is after the change; "
             f"reading took {ratio_mid:.0%} of the first time.")

### 2. The morning after the sync

In [ ]:
step("2. The morning after the sync")
SYNC_OLD = "# LANTERN case memory: inquiry EOW/2026/0412\n"
SYNC_NEW = SYNC_OLD + "Rewritten by the nightly sync, 31 July 2026 01:58.\n"
changes = {s.name: layout.how_often(s.name) for s in base}
p_save = learn.predict_number(
    "Put what changes least first: standing orders and tool schemas, then the weekly "
    "summaries and notes, then the case memory, then the session and the question. What share "
    "of the morning's first-question reading time does that save?", unit="%")
morning = {"today's order": [], "stable first": []}
yesterday = baseline.asking(base, questions[0]["question"])
overnight = changed_line(base, "case memory", SYNC_OLD, SYNC_NEW)
for _ in range(REPEATS):
    for name in morning:
        def order(parts, name=name):
            return parts if name == "today's order" else LAYOUT(parts, changes)[0]
        send(yesterday, order(yesterday))
        q1 = baseline.asking(overnight, questions[1]["question"])
        q2 = baseline.asking(overnight, questions[2]["question"])
        morning[name].append({"first": send(q1, order(q1)), "second": send(q2, order(q2))})
ordered, ends = LAYOUT(base, changes)
reread = {"today's order": total - before(base, "case memory"),
          "stable first": total - before(ordered, "case memory")}
table = Table(title="The morning after the sync: reading the prompt, seconds")
for col in ("layout", "tokens from the change on", *[f"first question, run {i + 1}"
                                                     for i in range(REPEATS)],
            "first, median", "second, median"):
    table.add_column(col, justify="left" if col == "layout" else "right")
mm = {}
for name, rs in morning.items():
    mm[name] = (statistics.median(r["first"]["seconds"] for r in rs),
                statistics.median(r["second"]["seconds"] for r in rs))
    table.add_row(name, f"{reread[name]:,}", *[f"{r['first']['seconds']:.2f}" for r in rs],
                  f"{mm[name][0]:.2f}", f"{mm[name][1]:.2f}")
console.print(table)
saved = 100 * (1 - mm["stable first"][0] / max(mm["today's order"][0], 1e-9))
note("Stable first: " + ", ".join(f"{s.name} ({layout.how_often(s.name)})" for s in ordered)
     + f". The cache breaks at indexes {ends}.")
learn.reveal(p_save, actual=round(saved), tolerance=0.3,
             why="Today the memory is second, so the sync's change makes the server read the "
             "notes, the summaries and all 19 tool schemas again. Stable first, the change is "
             "near the end and only what follows it is read again.")

### 3. Does the order change the answers?

In [ ]:
step("3. Does the order change the answers?")
p_score = learn.predict(
    "The case memory now travels as a user turn after the tools, not inside the system "
    "message. Asked the 26 signed questions, does LANTERN score differently from chapter 02's "
    "baseline of 17?", ["The same score, the same answers", "The same score, different answers",
                        "A different score"])
ref = baseline.reference(chat=chat)
parts = baseline.with_evidence(base)
results, seconds, outs = [], 0.0, []


def counted(*args, **kwargs):
    r = chat(*args, **kwargs)
    outs.append(r.output_tokens)
    return r


for q in questions:
    asked = baseline.asking(parts, q["question"])
    messages, schemas = layout.request(LAYOUT(asked, changes)[0])
    t0 = time.perf_counter()
    got = baseline.answer(messages, schemas, chat=counted, seed=baseline.SEED,
                          max_tokens=160)
    seconds += time.perf_counter() - t0
    results.append({"id": q["id"], "slice": q["slice"], "question": q["question"], **got,
                    "passed": baseline.grade(got["answer"], q["expect"])})
(OUT / "stable-first.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
diff = baseline.changed(ref, results)
passed, was = sum(r["passed"] for r in results), sum(r["passed"] for r in ref)
same_answers = sum(a["answer"] == b["answer"] for a, b in zip(ref, results, strict=True))
note(f"Stable first: {passed} of 26, against {was} of 26. Fixed: {diff['fixed'] or 'none'}. "
     f"Broken: {diff['broken'] or 'none'}. {same_answers} of 26 answers word for word the "
     f"same. The 26 questions took {seconds:.0f} s.")
learn.reveal(p_score, answer=2 if passed != was else (0 if same_answers == 26 else 1),
             why="Moving a section changes what the model reads next to what. Chapter 06 "
             "measured that place matters; a layout for the cache has to be scored like any "
             "other change to the context.")

### 4. On a hosted model: breakpoints and a price calculation

In [ ]:
step("4. On a hosted model: breakpoints and a price calculation")
kw = layout.hosted_request(ordered, ends, model=cfg["hosted"]["model"], max_tokens=160)
stable = sum(sizes[s.name] for s in ordered[:ends[-2] + 1])
tail = total - stable
out_tokens = round(sum(outs) / len(questions))
table = Table(title=f"A session of {SESSION} questions on {cfg['hosted']['model']}: a price "
              "calculation from these token counts, not a bill")
for col in ("questions arrive", "no cache", "cache, 5 minutes", "cache, 1 hour"):
    table.add_column(col, justify="left" if col == "questions arrive" else "right")
costs = {}
for label, writes_5m in (("within 5 minutes of each other", 1),
                         ("20 minutes apart", SESSION)):
    row = [layout.session_cost(stable=stable, tail=tail, output=out_tokens, questions=SESSION,
                               writes=0, prices=prices, cached=False),
           layout.session_cost(stable=stable, tail=tail, output=out_tokens, questions=SESSION,
                               writes=writes_5m, prices=prices, ttl="5m"),
           layout.session_cost(stable=stable, tail=tail, output=out_tokens, questions=SESSION,
                               writes=1, prices=prices, ttl="1h")]
    costs[label] = row
    table.add_row(label, *[f"${c:.4f}" for c in row])
console.print(table)
note(f"The request carries {layout.breakpoints_in(kw)} cache breakpoints, one at the end of "
     f"each group but the question. Cached prefix: everything up to the case memory, "
     f"{stable:,} tokens; after it, {tail:,}; {out_tokens} tokens out a question, as step 3 "
     "measured. Prices per "
     f"million tokens from config.yaml, checked {cfg['hosted']['prices_checked']}. Not run on "
     "the hosted model: no key.")
busy_end = others()

learn.record(
    context_tokens=total,
    middle_at=middle_at,
    repeats=REPEATS,
    first_s=med["first time"],
    same_s=med["the same again"],
    end_s=med["end changed"],
    middle_s=med["middle changed"],
    start_s=med["start changed"],
    same_count_reported=same_count,
    today_first_s=mm["today's order"][0],
    today_second_s=mm["today's order"][1],
    stable_first_s=mm["stable first"][0],
    stable_second_s=mm["stable first"][1],
    today_reread_tokens=reread["today's order"],
    stable_reread_tokens=reread["stable first"],
    morning_saved_pct=round(saved),
    breakpoints=layout.breakpoints_in(kw),
    stable_passed=passed,
    baseline_passed=was,
    fixed=len(diff["fixed"]),
    broken=len(diff["broken"]),
    same_answers=same_answers,
    cached_prefix_tokens=stable,
    session_no_cache_usd=round(costs["within 5 minutes of each other"][0], 6),
    session_5m_close_usd=round(costs["within 5 minutes of each other"][1], 6),
    session_5m_apart_usd=round(costs["20 minutes apart"][1], 6),
    session_1h_apart_usd=round(costs["20 minutes apart"][2], 6),
    other_runs_start=busy_start,
    other_runs_end=busy_end,
    mine=MINE is not None,
)

finding(f"""
On qwen3:8b on local Ollama, reading this {total:,}-token context took {med['first time']:.2f} s
the first time and {med['the same again']:.2f} s the same again (medians of {REPEATS}). A change
at the end cost {med['end changed']:.2f} s, in the middle {med['middle changed']:.2f} s, at the
start {med['start changed']:.2f} s: the server reads again from the first thing that changed,
and reports the same token count either way. On the morning after the sync, today's order made
the first question read {reread["today's order"]:,} tokens again
({mm["today's order"][0]:.2f} s); with what changes least first it was
{reread['stable first']:,} ({mm['stable first'][0]:.2f} s), {saved:.0f}% less time. The new
order scored {passed} of 26 against {was}, with {len(diff['fixed'])} fixed and
{len(diff['broken'])} broken. On the hosted model the same layout carries
{layout.breakpoints_in(kw)} cache breakpoints; its price is calculated here, not measured.
""")
save_shot("16-layout-for-the-cache")

## The nuances

- **The cost of a change is where it is.** Reading the context took 29.71 s the first time and
  0.06 s the same again. A changed question cost 0.40 s, a change 2,562 tokens in 15.36 s,
  about half, and a changed first line 29.65 s, the same as reading it cold.
- **Ollama counts the whole prompt whatever it reused.** The repeat reported 5,141 prompt
  tokens, as the first request did, and took 0.06 s. The saving shows in the time, never in
  the count. The run's own check printed `False` for "the same on every request" because the
  changed prompts are slightly different lengths (5,104 to 5,144 tokens), not because the
  count fell.
- **The first morning question is the one the layout is for.** On the first run, today's order
  read 4,576 tokens again from the rewritten case memory, in 23.28 s. Stable first, 1,150
  tokens, in 6.73 s.
- **Only that first run measured the morning.** Runs 2 and 3 of today's order took 0.34 s
  each: Ollama still held the rewritten beginning from run 1, through four other requests in
  between. So the median for today's order, 0.34 s, and the saving the run prints from it,
  -1909%, do not measure the sync's cost. The stable first runs did not get that reuse (6.73,
  6.83 and 7.38 s). Which beginnings a server keeps, and for how long, is its own business; a
  measurement of the cache has to clear it or show that it did.
- **The machine was shared.** Other lessons were running when the run started (4) and when it
  ended (7). Every timing is a median of three for that reason, and the seconds here belong to
  this machine on this morning.
- **Moving a section moves answers.** Stable first, LANTERN scored 16 of 26 against the
  baseline's 17. Q-10 broke: with the case memory now a user turn after the tools, LANTERN
  called `lookup_company`, which is not connected, for an answer the memory holds. Only 8 of
  26 answers were word for word the same. Chapter 06 found that place matters; a layout for the
  cache is a change to the context and is scored like one.
- **A cached prefix has to be paid for first.** On the hosted model a cache write costs more
  than plain input (2.50 against 2.00 dollars per million tokens for five minutes). Questions
  twenty minutes apart with a five-minute cache rewrite the prefix every time, and the session
  costs more than with no cache at all.

## Try this

1. Put the date at the top of the standing orders and send the same request twice. What does
   the second one cost?
2. Make step 2 send an unrelated long request before each morning question, and see whether
   today's order then takes the first run's time on every run.
3. Pass `"case memory": "per turn"` to `layout.layout` and look at where the breakpoints go.

## Master it

After this chapter you can measure what a change costs by where it sits in the context, order a
context so what changes least comes first, mark where a hosted request's cache breakpoints go,
price a session from token counts, and score a new layout before keeping it.

**Predict, then see.** `make lesson N=16` stops four times: what Ollama reports as a repeated
request's token count, what a change in the middle costs, what share of the morning the layout
saves, and whether the score changes.

**Build it.** Write `layout` in [`practice/exercise.py`](practice/exercise.py):

```bash
make exercise N=16          # 7 offline tests
make hint N=16
make lesson N=16 ARGS=--mine
```

**Check yourself.** `make quiz N=16`: six questions.

**Break it.** `make challenge N=16`: your layout on the morning; a morning measured from a
cleared cache; and a timestamp at the top of the context.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 16

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 16

In [ ]:
%run ../../scripts/learn.py hint 16

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 16

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

On qwen3:8b on local Ollama, reading LANTERN's 5,410-token context took 29.71 s cold and 0.06 s
the same again. A change cost what followed it: 0.40 s at the end, 15.36 s in the middle,
29.65 s at the start. The nightly sync rewrites the second section, so on the first measured
morning the first question read 4,576 tokens again in 23.28 s; with what changes least first
it read 1,150 in 6.73 s. The repeats of that morning were served from a beginning the server
had kept, so the run's median saving is not a measurement, and the chapter says so. The new
order scored 16 of 26 against 17 and broke Q-10, so it is not adopted on speed alone. Priced
for the hosted model, the same layout makes a ten-question session $0.0353 with a five-minute
cache against $0.1144 without, as a calculation. The sync's 01:58 write now has a cost in
seconds as well as in what it wrote.

**Next:** [Chapter 17: Four ways to be wrong](../17-how-context-fails/README.md)